In [ ]:
#1
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    f1_score
)

In [ ]:
#2
clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")

print("CLASSIFICATION DATASET")
display(clf_df.head())

print("REGRESSION DATASET")
display(reg_df.head())

print("Classification shape:", clf_df.shape)
print("Regression shape:", reg_df.shape)

CLASSIFICATION DATASET


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


REGRESSION DATASET


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


Classification shape: (900, 7)
Regression shape: (800, 7)


In [ ]:
#3
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

numeric_features = X_clf.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_clf.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']


C:\Users\bisra\AppData\Local\Temp\ipykernel_3644\2816144480.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_clf.select_dtypes(


In [ ]:
#4
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

print("Training size:", X_train_clf.shape)
print("Testing size:", X_test_clf.shape)

Training size: (720, 5)
Testing size: (180, 5)


In [38]:
classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    )
}

clf_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [39]:
clf_results = []

for name, model in classifiers.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train_clf,
        y_train_clf,
        cv=clf_cv,
        scoring="f1",
        return_train_score=False
    )

    clf_results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

clf_results_df = pd.DataFrame(clf_results)

display(
    clf_results_df.sort_values(
        by="Mean F1",
        ascending=False
    )
)

,Model,Mean F1,Std F1
0,Logistic Regression,0.392651,0.062747
1,KNN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


In [40]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

clf_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

clf_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=clf_param_grid,
    scoring="f1",
    cv=clf_cv,
    n_jobs=-1
)

clf_search.fit(X_train_clf, y_train_clf)

print("Best parameters:", clf_search.best_params_)
print("Best CV F1:", clf_search.best_score_)

Best parameters: {'model__C': 10}
Best CV F1: 0.4225932078257659


In [41]:
best_clf = clf_search.best_estimator_

y_pred_clf = best_clf.predict(X_test_clf)

print("Test F1 Score:")
print(f1_score(y_test_clf, y_pred_clf))

print("\nClassification Report:")
print(classification_report(y_test_clf, y_pred_clf))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test_clf, y_pred_clf)
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nFalse Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)
print("True Negatives:", tn)

Test F1 Score:
0.3137254901960784

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180


Confusion Matrix:
[[137   5]
 [ 30   8]]

False Positives: 5
False Negatives: 30
True Positives: 8
True Negatives: 137


Experiment B — Regression

In [42]:
#10
X_reg = reg_df.drop(
    columns=["ticket_id", "resolution_time_hours"]
)

y_reg = reg_df["resolution_time_hours"]

numeric_features_reg = X_reg.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_reg = X_reg.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", numeric_features_reg)
print("Categorical features:", categorical_features_reg)

Numeric features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']


C:\Users\bisra\AppData\Local\Temp\ipykernel_3644\3137527175.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_reg = X_reg.select_dtypes(


In [51]:
#11
numeric_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer(transformers=[
    ("num", numeric_transformer_reg, numeric_features_reg),
    ("cat", categorical_transformer_reg, categorical_features_reg)
])

In [44]:
#12
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training size:", X_train_reg.shape)
print("Testing size:", X_test_reg.shape)

Training size: (640, 5)
Testing size: (160, 5)


In [45]:
#13
regressors = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(),

    "Decision Tree Regression": DecisionTreeRegressor(
        random_state=42
    )
}

reg_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [46]:
#14
reg_results = []

for name, model in regressors.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_reg),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train_reg,
        y_train_reg,
        cv=reg_cv,
        scoring="neg_mean_absolute_error",
        return_train_score=False
    )

    mae_scores = -scores["test_score"]

    reg_results.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

reg_results_df = pd.DataFrame(reg_results)

display(
    reg_results_df.sort_values(
        by="Mean MAE",
        ascending=True
    )
)

,Model,Mean MAE,Std MAE
1,Ridge Regression,2.035127,0.171404
0,Linear Regression,2.035134,0.171650
2,Decision Tree Regression,3.239097,0.053807


In [47]:
#15
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

reg_param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100, 1000]
}

reg_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=reg_param_grid,
    scoring="neg_mean_absolute_error",
    cv=reg_cv,
    n_jobs=-1
)

reg_search.fit(X_train_reg, y_train_reg)

print("Best parameters:", reg_search.best_params_)
print("Best CV MAE:", -reg_search.best_score_)

Best parameters: {'model__alpha': 0.1}
Best CV MAE: 2.035118085555478


In [48]:
#16
best_reg = reg_search.best_estimator_

y_pred_reg = best_reg.predict(X_test_reg)

test_mae = mean_absolute_error(
    y_test_reg,
    y_pred_reg
)

test_rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_reg)
)

print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)

Test MAE: 2.230086144180656
Test RMSE: 3.4027561084563143


In [49]:
#17
Q1 = y_reg.quantile(0.25)
Q3 = y_reg.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = reg_df[
    (y_reg < lower_bound) |
    (y_reg > upper_bound)
]

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

print("Number of Outliers:", len(outliers))

display(outliers.head())


Lower Bound: -5.940557578774017
Upper Bound: 16.917779901664556
Number of Outliers: 10


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
19,R0019,4.0,14.0,High,Phone,38.541669,17.981195
49,R0049,5.0,10.0,High,Chat,46.570515,24.108510
51,R0051,7.0,3.0,Low,Phone,83.017767,23.929861
113,R0113,8.0,1.0,Critical,Chat,82.259644,18.267015
188,R0188,8.0,11.0,Medium,Phone,4.134498,26.135421


In [50]:
#18
print("FINAL CLASSIFICATION MODEL")
print("Selected Model: Tuned Logistic Regression")
print("Best Parameters:", clf_search.best_params_)
print("Test F1:", f1_score(y_test_clf, y_pred_clf))

print("\nFINAL REGRESSION MODEL")
print("Selected Model: Tuned Ridge Regression")
print("Best Parameters:", reg_search.best_params_)
print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)

FINAL CLASSIFICATION MODEL
Selected Model: Tuned Logistic Regression
Best Parameters: {'model__C': 10}
Test F1: 0.3137254901960784

FINAL REGRESSION MODEL
Selected Model: Tuned Ridge Regression
Best Parameters: {'model__alpha': 0.1}
Test MAE: 2.230086144180656
Test RMSE: 3.4027561084563143
